# NFLverse Bronze Source Inventory & EDA

## tl;dr

All seven candidate sources loaded successfully. Candidate grains had no duplicate rows; weekly player stats had 66 null player keys and weekly rosters had 30 null GSIS keys.

Player-stat and PBP receiver IDs matched the player master 100%, weekly roster IDs matched 99.59%, snap-count PFR IDs matched 99.86%, and 2025 participation game/plays matched PBP 100%. The 2025 PBP pandas DataFrame used about 370 MB, supporting season-at-a-time ingestion.

The core WR Bronze sources are weekly player stats, players, schedules, weekly rosters, and play-by-play. Snap counts are useful with crosswalk validation; participation remains optional. No Bronze files are written and no Silver or Gold transformations are defined here.

## Context & Methods

### Scope

- Seasons: 2023–2025
- First vertical slice: wide receivers
- Environment: `sports_dev_env`
- In-memory workflow: `nflreadpy` Polars output converted explicitly to pandas
- Large-source first pass: 2025 only for play-by-play and participation

The objective is to first inspect the available NFL data. Then we decide which datasets and columns are worth saving as raw Bronze Parquet files.  

This inventory notebook helps us avoid downloading and storing unnecessary sources. It also confirms that the selected data covers 2023–2025 and contains the identifiers and WR fields we need.

## Setup

In [1]:
import nflreadpy as nfl
import pandas as pd
from eda_utils import column_summary

SEASONS = [2023, 2024, 2025]
LARGE_SOURCE_SEASONS = [2025]

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 180)

## Data

## 1. Players

### What we are verifying

- Available player identifiers and metadata
- Whether `gsis_id` is a usable player key
- Availability of the `pfr_id` crosswalk needed by snap counts
- Position coverage
- Null and duplicate identifiers
- Fields useful for WR identity and display

In [2]:
players = nfl.load_players().to_pandas()
players.head()

,gsis_id,display_name,common_first_name,first_name,last_name,short_name,football_name,suffix,esb_id,nfl_id,pfr_id,pff_id,otc_id,espn_id,smart_id,...,college_conference,jersey_number,rookie_season,last_season,latest_team,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
0,00-0028830,Isaako Aaitui,Isaako,Isaako,Aaitui,None,None,None,AAI622937,None,AaitIs00,6998,2535,14856,32004141-4962-2937-61ff-017b1804dec6,...,None,0,2011,2014,WAS,DEV,None,None,2,DI,None,NaN,NaN,NaN,None
1,00-0038389,Israel Abanikanda,Israel,Israel,Abanikanda,I.Abanikanda,Israel,None,ABA159567,56008,AbanIs00,122999,10967,4429202,32004142-4115-9567-2e24-0eab29f6a4b9,...,Atlantic Coast Conference,25,2023,2026,DAL,DEV,DEV,Practice Squad,3,HB,A,2023.0,5.0,143.0,NYJ
2,00-0024644,Jon Abbate,Jon,Jon,Abbate,None,None,None,ABB051371,None,None,None,None,None,32004142-4205-1371-db95-1abc96313b69,...,None,67,2007,2007,HOU,RES,None,None,0,None,None,NaN,NaN,NaN,None
3,ABB498348,Vince Abbott,Vince,Vincent,Abbott,None,None,None,ABB498348,None,abbotvin01,None,None,None,32004142-4249-8348-e00f-5fbbe6a0c73c,...,None,0,1987,1988,LAC,ACT,None,None,2,None,None,NaN,NaN,NaN,None
4,00-0031021,Jared Abbrederis,Jared,Jared,Abbrederis,J.Abbrederis,Jared,None,ABB650964,41405,AbbrJa00,8811,3115,16836,32004142-4265-0964-fc36-bb0ad76ff6e6,...,None,10,2014,2017,DET,CUT,CUT,None,4,WR,None,2014.0,5.0,176.0,GB


In [3]:
players_summary = column_summary(players)
players_summary

,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,ngs_position_group,object,24828,4984,19844,79.93,9,0.18,0.04,False,False,"[RB, WR, DL, DB, OL, TE, LB, QB, SPEC]",categorical
1,position_group,object,24828,24828,0,0.00,9,0.04,0.04,False,False,"[DL, RB, LB, SPEC, WR, DB, TE, OL, QB]",categorical
2,suffix,object,24828,164,24664,99.34,9,5.49,0.04,False,False,"[Jr., III, II, IV, Jr, Sindre, Sr., , Williams]",categorical
3,pff_status,object,24828,3885,20943,84.35,12,0.31,0.05,False,False,>10 values,categorical
4,status,object,24828,24828,0,0.00,13,0.05,0.05,False,False,>10 values,categorical
5,ngs_position,object,24828,4896,19932,80.28,17,0.35,0.07,False,False,>10 values,categorical
6,pff_position,object,24828,7452,17376,69.99,17,0.23,0.07,False,False,>10 values,categorical
7,ngs_status,object,24828,8292,16536,66.60,20,0.24,0.08,False,False,>10 values,categorical
8,draft_round,float64,24828,12490,12338,49.69,17,0.14,0.07,False,False,>10 values,low_cardinality_numeric
9,height,float64,24828,24818,10,0.04,20,0.08,0.08,False,False,>10 values,low_cardinality_numeric


In [4]:
players_key_check = pd.DataFrame(
    {
        "check": [
            "rows",
            "null gsis_id",
            "duplicate non-null gsis_id",
            "null pfr_id",
            "duplicate non-null pfr_id",
        ],
        "value": [
            len(players),
            players["gsis_id"].isna().sum(),
            players.loc[players["gsis_id"].notna(), "gsis_id"].duplicated().sum(),
            players["pfr_id"].isna().sum(),
            players.loc[players["pfr_id"].notna(), "pfr_id"].duplicated().sum(),
        ],
    }
)

players_key_check

,check,value
0,rows,24828
1,null gsis_id,0
2,duplicate non-null gsis_id,0
3,null pfr_id,2175
4,duplicate non-null pfr_id,0


In [5]:
# number of players at each position
players["position"].value_counts(dropna=False).head(20)

position
WR     3248
LB     2610
RB     2451
DB     2309
DE     1754
OT     1694
TE     1583
G      1577
DT     1480
CB     1207
QB     1006
C       785
OLB     471
K       378
P       350
SAF     334
S       325
FS      282
NT      246
FB      210
Name: count, dtype: int64

In [6]:
players[["gsis_id", "pfr_id", "display_name", "position", "position_group", "latest_team"]].head(10)

,gsis_id,pfr_id,display_name,position,position_group,latest_team
0,00-0028830,AaitIs00,Isaako Aaitui,NT,DL,WAS
1,00-0038389,AbanIs00,Israel Abanikanda,RB,RB,DAL
2,00-0024644,None,Jon Abbate,LB,LB,HOU
3,ABB498348,abbotvin01,Vince Abbott,K,SPEC,LAC
4,00-0031021,AbbrJa00,Jared Abbrederis,WR,WR,DET
5,00-0032860,AbdeMe00,Mehdi Abdesmad,DE,DL,TEN
6,00-0028564,AbduIs00,Isa Abdul-Quddus,S,DB,MIA
7,00-0032104,AbduAm00,Ameer Abdullah,RB,RB,JAX
8,00-0023663,AbduHa20,Hamza Abdullah,DB,DB,ARI
9,00-0025940,AbduHu99,Husain Abdullah,FS,DB,KC


## 2. Schedules

### What we are verifying

- Coverage for 2023–2025
- Regular-season and postseason representation
- Whether `game_id` uniquely identifies a game
- Week ranges by season and game type
- Home and away team coverage
- Game identifiers needed to connect weekly and play-level sources

In [7]:
schedules = nfl.load_schedules(seasons=SEASONS).to_pandas()
schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,...,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2023_01_DET_KC,2023,REG,1,2023-09-07,Thursday,20:20,DET,21,KC,20,Home,-1,41,0,...,-110,0,outdoors,,NaN,NaN,00-0033106,00-0033873,Jared Goff,Patrick Mahomes,Dan Campbell,Andy Reid,John Hussey,KAN00,GEHA Field at Arrowhead Stadium
1,2023_01_CAR_ATL,2023,REG,1,2023-09-10,Sunday,13:00,CAR,10,ATL,24,Home,14,34,0,...,-110,1,closed,,NaN,NaN,00-0039150,00-0038122,Bryce Young,Desmond Ridder,Frank Reich,Arthur Smith,Brad Rogers,ATL97,Mercedes-Benz Stadium
2,2023_01_HOU_BAL,2023,REG,1,2023-09-10,Sunday,13:00,HOU,9,BAL,25,Home,16,34,0,...,-110,0,outdoors,,NaN,NaN,00-0039163,00-0034796,C.J. Stroud,Lamar Jackson,DeMeco Ryans,John Harbaugh,Tra Blake,BAL00,M&T Bank Stadium
3,2023_01_CIN_CLE,2023,REG,1,2023-09-10,Sunday,13:00,CIN,3,CLE,24,Home,21,27,0,...,-110,1,outdoors,,NaN,NaN,00-0036442,00-0033537,Joe Burrow,Deshaun Watson,Zac Taylor,Kevin Stefanski,Clete Blakeman,CLE00,FirstEnergy Stadium
4,2023_01_JAX_IND,2023,REG,1,2023-09-10,Sunday,13:00,JAX,31,IND,21,Home,-10,52,0,...,-110,1,closed,,NaN,NaN,00-0036971,00-0039164,Trevor Lawrence,Anthony Richardson,Doug Pederson,Shane Steichen,Clay Martin,IND00,Lucas Oil Stadium


In [8]:
schedules_summary = column_summary(schedules)
schedules_summary

,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,nfl_detail_id,object,855,0,855,100.00,0,0.00,0.00,False,False,[],binary_categorical
1,location,object,855,855,0,0.00,2,0.23,0.23,False,False,"[Home, Neutral]",binary_categorical
2,div_game,int32,855,855,0,0.00,2,0.23,0.23,False,False,"[0, 1]",binary_numeric
3,overtime,int32,855,855,0,0.00,2,0.23,0.23,False,False,"[0, 1]",binary_numeric
4,roof,object,855,855,0,0.00,4,0.47,0.47,False,False,"[outdoors, closed, dome, open]",categorical
5,game_type,object,855,855,0,0.00,5,0.58,0.58,False,False,"[REG, WC, DIV, CON, SB]",categorical
6,weekday,object,855,855,0,0.00,6,0.70,0.70,False,False,"[Thursday, Sunday, Monday, Friday, Saturday, Wednesday]",categorical
7,surface,object,855,855,0,0.00,7,0.82,0.82,False,False,"[, grass, fieldturf, sportturf, a_turf, astroturf, matrixturf]",categorical
8,gametime,object,855,855,0,0.00,18,2.11,2.11,False,False,>10 values,categorical
9,referee,object,855,855,0,0.00,18,2.11,2.11,False,False,>10 values,categorical


In [9]:
schedule_key_check = pd.DataFrame(
    {
        "check": ["rows", "null game_id", "duplicate game_id"],
        "value": [
            len(schedules),
            schedules["game_id"].isna().sum(),
            schedules["game_id"].duplicated().sum(),
        ],
    }
)
schedule_key_check

,check,value
0,rows,855
1,null game_id,0
2,duplicate game_id,0


In [10]:
# Week ranges by season and game type
(schedules.groupby(["season", "game_type"], dropna=False)
            .agg(
                    games=("game_id", "size"),
                    distinct_games=("game_id", "nunique"),
                    min_week=("week", "min"),
                    max_week=("week", "max"),
                ).reset_index()
)

,season,game_type,games,distinct_games,min_week,max_week
0,2023,CON,2,2,21,21
1,2023,DIV,4,4,20,20
2,2023,REG,272,272,1,18
3,2023,SB,1,1,22,22
4,2023,WC,6,6,19,19
5,2024,CON,2,2,21,21
6,2024,DIV,4,4,20,20
7,2024,REG,272,272,1,18
8,2024,SB,1,1,22,22
9,2024,WC,6,6,19,19


In [11]:
# Home and away team coverage
(pd.DataFrame(
        {
            "home_teams": sorted(schedules["home_team"].dropna().unique()),
            "away_teams": sorted(schedules["away_team"].dropna().unique()),
        }
    ).head(40)
)

,home_teams,away_teams
0,ARI,ARI
1,ATL,ATL
2,BAL,BAL
3,BUF,BUF
4,CAR,CAR
5,CHI,CHI
6,CIN,CIN
7,CLE,CLE
8,DAL,DAL
9,DEN,DEN


## 3. Weekly Player Stats

### What we are verifying

- Coverage for 2023–2025
- Regular-season and postseason rows
- Candidate grain: player, season, week, and season type
- Null player identifiers and what those rows represent
- Team and position coverage
- Directly available WR volume, production, air-yard, and fantasy fields
- Player-ID match rate to the player master

In [12]:
player_stats = nfl.load_player_stats(
    seasons=SEASONS,
    summary_level="week",
).to_pandas()

player_stats.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,...,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,"https://static.www.nfl.com/image/upload/f_auto,q_auto/league/dypvakakxhccxs67tb0y",2023,1,REG,2023_01_BUF_NYJ,NYJ,BUF,0,1,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
1,00-0023853,M.Prater,Matt Prater,K,SPEC,"https://static.www.nfl.com/image/upload/f_auto,q_auto/league/pj981bi535y4jwy6skr1",2023,1,REG,2023_01_ARI_WAS,ARI,WAS,0,0,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
2,00-0025565,N.Folk,Nick Folk,K,SPEC,"https://static.www.nfl.com/image/upload/f_auto,q_auto/league/qf6vaghxuoxqw0pyrvg3",2023,1,REG,2023_01_TEN_NO,TEN,NO,0,0,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0026190,C.Campbell,Calais Campbell,DE,DL,"https://static.www.nfl.com/image/upload/f_auto,q_auto/league/ekkaync0yveo2cwxlljr",2023,1,REG,2023_01_CAR_ATL,ATL,CAR,0,0,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
4,00-0026498,M.Stafford,Matthew Stafford,QB,QB,"https://static.www.nfl.com/image/upload/f_auto,q_auto/league/jwpkjfrkzufdyh8u1mg7",2023,1,REG,2023_01_LA_SEA,LA,SEA,24,38,334,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,14.46,14.46


In [13]:
player_stats_summary = column_summary(player_stats)
player_stats_summary

,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,season_type,object,57048,57048,0,0.00,2,0.00,0.00,False,False,"[REG, POST]",binary_categorical
1,fg_missed_0_19,int32,57048,57048,0,0.00,1,0.00,0.00,True,False,[0],binary_numeric
2,def_2pt_atts,int32,57048,57048,0,0.00,2,0.00,0.00,False,False,"[0, 1]",binary_numeric
3,def_2pt_made,int32,57048,57048,0,0.00,2,0.00,0.00,False,False,"[0, 1]",binary_numeric
4,def_fg_blocks,int32,57048,57048,0,0.00,2,0.00,0.00,False,False,"[0, 1]",binary_numeric
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,fg_made_list,object,57048,1423,55625,97.51,859,60.37,1.51,False,False,>10 values,text_or_high_cardinality_categorical
146,player_name,object,57048,56982,66,0.12,2575,4.52,4.51,False,False,>10 values,text_or_high_cardinality_categorical
147,player_display_name,object,57048,56982,66,0.12,2829,4.96,4.96,False,False,>10 values,text_or_high_cardinality_categorical
148,headshot_url,object,57048,56968,80,0.14,2830,4.97,4.96,False,False,>10 values,text_or_high_cardinality_categorical


In [14]:
# check nulls and duplicate candidate grain (player, season, week, and season type)
player_stats_key = ["player_id", "season", "week", "season_type"]
player_stats_duplicate_rows = player_stats.duplicated(player_stats_key).sum()
player_stats_null_player_ids = player_stats["player_id"].isna().sum()

display(
    pd.DataFrame(
        {
            "check": [
                "rows",
                "null player_id",
                "duplicate candidate grain",
            ],
            "value": [
                len(player_stats),
                player_stats_null_player_ids,
                player_stats_duplicate_rows,
            ],
        }
    )
)

,check,value
0,rows,57048
1,null player_id,66
2,duplicate candidate grain,0


In [15]:
display(
    player_stats.groupby(["season", "season_type"], dropna=False)
    .agg(
        rows=("season", "size"),
        players=("player_id", "nunique"),
        teams=("team", "nunique"),
        min_week=("week", "min"),
        max_week=("week", "max"),
    )
    .reset_index()
)

,season,season_type,rows,players,teams,min_week,max_week
0,2023,POST,837,498,14,19,22
1,2023,REG,17806,1942,32,1,18
2,2024,POST,853,496,14,19,22
3,2024,REG,18130,1996,32,1,18
4,2025,POST,882,528,14,19,22
5,2025,REG,18540,2019,32,1,18


In [16]:
display(player_stats["position"].value_counts(dropna=False).head(20))

position
LB     8090
WR     7795
CB     6155
RB     4799
DT     4760
DE     4634
SAF    3920
TE     3860
QB     2079
K      1707
P      1680
OT     1448
DB     1357
G       900
OLB     756
FS      689
C       439
MLB     403
S       358
ILB     289
Name: count, dtype: int64

In [17]:
# Looking at the empty player_ids
# Looks like theres no actual stats recorded for these. Maybe we could clean these rows out in data pipeline 
display(
    player_stats.loc[
        player_stats["player_id"].isna(),
        ["season", "week", "season_type", "team", "player_id", "player_name", "position", "targets", "receptions"],
    ]# .head(20)
)

,season,week,season_type,team,player_id,player_name,position,targets,receptions
1057,2023,1,REG,DET,None,None,None,0,0
2073,2023,2,REG,PHI,None,None,None,0,0
3110,2023,3,REG,SF,None,None,None,0,0
4184,2023,4,REG,DET,None,None,None,0,0
5091,2023,5,REG,CHI,None,None,None,0,0
6070,2023,6,REG,DEN,None,None,None,0,0
6910,2023,7,REG,JAX,None,None,None,0,0
7962,2023,8,REG,TB,None,None,None,0,0
8908,2023,9,REG,PIT,None,None,None,0,0
9833,2023,10,REG,CHI,None,None,None,0,0


In [18]:
# check directly available WR volume, production, air-yard, and fantasy fields

wr_stat_columns = [
    "season",
    "week",
    "season_type",
    "player_id",
    "player_display_name",
    "team",
    "position",
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "receiving_air_yards",
    "target_share",
    "air_yards_share",
    "fantasy_points_ppr",
]

missing_wr_stat_columns = [
    column for column in wr_stat_columns if column not in player_stats.columns
]
print("Missing candidate WR columns:", missing_wr_stat_columns)

display(
    player_stats.loc[
        (player_stats["season"] == 2025)
        & (player_stats["season_type"] == "REG")
        & (player_stats["position"] == "WR"),
        [column for column in wr_stat_columns if column in player_stats.columns],
    ]
    .sort_values("targets", ascending=False)
    .head(15)
)

player_stats_with_id = player_stats.loc[player_stats["player_id"].notna()]
player_stats_player_match_rate = player_stats_with_id["player_id"].isin(
    players["gsis_id"]
).mean()

print(f"Player stats → players GSIS match rate: {player_stats_player_match_rate:.2%}")

Missing candidate WR columns: []


,season,week,season_type,player_id,player_display_name,team,position,targets,receptions,receiving_yards,receiving_tds,receiving_air_yards,target_share,air_yards_share,fantasy_points_ppr
44434,2025,7,REG,00-0036900,Ja'Marr Chase,CIN,WR,23,16,161,1,160,0.511111,0.474777,38.1
45404,2025,8,REG,00-0036900,Ja'Marr Chase,CIN,WR,19,12,91,0,98,0.575758,0.453704,21.1
48469,2025,11,REG,00-0038559,Michael Wilson,ARI,WR,18,15,185,0,240,0.321429,0.618557,33.5
52350,2025,15,REG,00-0036963,Amon-Ra St. Brown,DET,WR,18,13,164,2,134,0.473684,0.386167,41.4
37941,2025,1,REG,00-0035662,Marquise Brown,KC,WR,16,10,99,0,101,0.421053,0.372694,19.9
39178,2025,2,REG,00-0036900,Ja'Marr Chase,CIN,WR,16,14,165,1,112,0.372093,0.311111,36.5
53469,2025,16,REG,00-0037239,Chris Olave,NO,WR,16,10,148,2,258,0.340426,0.611374,36.8
53709,2025,16,REG,00-0039075,Puka Nacua,LA,WR,16,12,225,2,171,0.333333,0.358491,46.5
51530,2025,14,REG,00-0038559,Michael Wilson,ARI,WR,16,11,142,2,193,0.363636,0.548295,37.2
52327,2025,15,REG,00-0036900,Ja'Marr Chase,CIN,WR,16,10,132,0,157,0.444444,0.552817,23.2


Player stats → players GSIS match rate: 100.00%


## 4. Weekly Rosters

### What we are verifying

- Coverage for 2023–2025
- Candidate grain: GSIS player ID, season, week, and team
- Null GSIS identifiers and what those rows represent
- Team, position, roster-status, and week coverage
- GSIS match rate to the player master
- Availability of alternate IDs useful for later joins

In [19]:
rosters_weekly = nfl.load_rosters_weekly(seasons=SEASONS).to_pandas()
rosters_weekly.head()

,season,team,position,depth_chart_position,jersey_number,status,full_name,first_name,last_name,birth_date,height,weight,college,gsis_id,espn_id,...,sleeper_id,years_exp,headshot_url,ngs_position,week,game_type,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number
0,2023,PHI,OL,T,74.0,CUT,Bernard Williams,Bernard,Williams,NaT,80.0,286.0,None,00-0017724,None,...,None,29,None,None,11,REG,W03,Bernard,WIL148626,17623,32005749-4c14-8626-f883-08eba7248da6,1994,1994.0,PHI,14.0
1,2023,SEA,OL,T,70.0,DEV,Jason Peters,Jason,Peters,1982-01-22,76.0,328.0,Arkansas,00-0022531,6012,...,412,19,"https://static.www.nfl.com/image/private/f_auto,q_auto/league/ma7bkdymewzelolhydoc",None,6,REG,P07,Jason,PET150539,29550,32005045-5415-0539-96c0-31361312950f,2004,2004.0,None,NaN
2,2023,SEA,OL,T,70.0,DEV,Jason Peters,Jason,Peters,1982-01-22,76.0,328.0,Arkansas,00-0022531,6012,...,412,19,"https://static.www.nfl.com/image/private/f_auto,q_auto/league/ma7bkdymewzelolhydoc",None,4,REG,P07,Jason,PET150539,29550,32005045-5415-0539-96c0-31361312950f,2004,2004.0,None,NaN
3,2023,SEA,OL,T,70.0,INA,Jason Peters,Jason,Peters,1982-01-22,76.0,328.0,Arkansas,00-0022531,6012,...,412,19,"https://static.www.nfl.com/image/upload/f_auto,q_auto/league/xcsyqvrlrpe6s3d0rodq",None,17,REG,A01,Jason,PET150539,29550,32005045-5415-0539-96c0-31361312950f,2004,2004.0,None,NaN
4,2023,SEA,OL,T,70.0,ACT,Jason Peters,Jason,Peters,1982-01-22,76.0,328.0,Arkansas,00-0022531,6012,...,412,19,"https://static.www.nfl.com/image/private/f_auto,q_auto/league/ma7bkdymewzelolhydoc",T,11,REG,A01,Jason,PET150539,29550,32005045-5415-0539-96c0-31361312950f,2004,2004.0,None,NaN


In [20]:
rosters_weekly_summary = column_summary(rosters_weekly)
rosters_weekly_summary

,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,game_type,object,139083,139083,0,0.00,5,0.00,0.00,False,False,"[REG, WC, CON, DIV, SB]",categorical
1,position,object,139083,139083,0,0.00,11,0.01,0.01,False,False,>10 values,categorical
2,status,object,139083,139083,0,0.00,13,0.01,0.01,False,False,>10 values,categorical
3,ngs_position,object,139083,64766,74317,53.43,16,0.02,0.01,False,False,>10 values,categorical
4,birth_date,datetime64[ms],139083,138473,610,0.44,2549,1.84,1.83,False,False,>10 values,datetime
5,season,int32,139083,139083,0,0.00,3,0.00,0.00,False,False,"[2023, 2024, 2025]",low_cardinality_numeric
6,height,float64,139083,138876,207,0.15,19,0.01,0.01,False,False,>10 values,low_cardinality_numeric
7,week,int32,139083,139083,0,0.00,22,0.02,0.02,False,False,>10 values,numerical
8,entry_year,int32,139083,139083,0,0.00,26,0.02,0.02,False,False,>10 values,numerical
9,rookie_year,float64,139083,139075,8,0.01,26,0.02,0.02,False,False,>10 values,numerical


In [21]:
roster_key = ["gsis_id", "season", "week", "team"]
roster_duplicate_rows = rosters_weekly.duplicated(roster_key).sum()
roster_null_gsis_ids = rosters_weekly["gsis_id"].isna().sum()

display(
    pd.DataFrame(
        {
            "check": [
                "rows",
                "null gsis_id",
                "duplicate candidate grain",
            ],
            "value": [
                len(rosters_weekly),
                roster_null_gsis_ids,
                roster_duplicate_rows,
            ],
        }
    )
)

,check,value
0,rows,139083
1,null gsis_id,30
2,duplicate candidate grain,0


In [22]:
display(
    rosters_weekly.groupby("season")
    .agg(
        rows=("season", "size"),
        players=("gsis_id", "nunique"),
        teams=("team", "nunique"),
        min_week=("week", "min"),
        max_week=("week", "max"),
    )
    .reset_index()
)

,season,rows,players,teams,min_week,max_week
0,2023,45655,3089,32,1,22
1,2024,46579,3215,32,1,22
2,2025,46849,3134,32,1,22


In [23]:
# positions counts
display(rosters_weekly["position"].value_counts(dropna=False).head(20))

position
DB    26531
OL    24742
DL    20790
LB    18526
WR    17058
RB    10341
TE     9258
QB     5902
K      2178
LS     1888
P      1869
Name: count, dtype: int64

In [24]:
# roster status
display(rosters_weekly["status"].value_counts(dropna=False).head(20))

status
ACT    82106
DEV    26108
RES    16142
INA    10773
CUT     2846
RET     1009
EXE       50
TRC       23
TRD       15
E14        4
TRT        3
E01        3
PUP        1
Name: count, dtype: int64

In [25]:
# sample of rows with no gsis_id
display(
    rosters_weekly.loc[
        rosters_weekly["gsis_id"].isna(),
        ["season", "week", "gsis_id", "team", "full_name", "position", "status", "pfr_id"],
    ].head(30)
)

rosters_with_id = rosters_weekly.loc[rosters_weekly["gsis_id"].notna()]
roster_player_match_rate = rosters_with_id["gsis_id"].isin(players["gsis_id"]).mean()

print(f"Weekly rosters → players GSIS match rate: {roster_player_match_rate:.2%}")

,season,week,gsis_id,team,full_name,position,status,pfr_id
45650,2023,18,None,DEN,Durell Nchami,LB,DEV,None
45651,2023,14,None,DEN,Durell Nchami,LB,DEV,None
45652,2023,15,None,DEN,Durell Nchami,LB,DEV,None
45653,2023,16,None,DEN,Durell Nchami,LB,DEV,None
45654,2023,17,None,DEN,Durell Nchami,LB,DEV,None
92227,2024,6,None,IND,Jack Wilson,OL,DEV,None
92228,2024,8,None,IND,Jack Wilson,OL,DEV,None
92229,2024,7,None,IND,Jack Wilson,OL,DEV,None
92230,2024,15,None,NO,Tra Fluellen,DB,DEV,None
92231,2024,18,None,NO,Tra Fluellen,DB,DEV,None


Weekly rosters → players GSIS match rate: 99.59%


## 5. Snap Counts

### What we are verifying

- Coverage for 2023–2025
- Candidate grain: PFR player ID, game, and team
- Null and duplicate key behavior
- Team, week, and position coverage
- Plausible offensive snap values
- Match rate from `pfr_player_id` to the player master `pfr_id`

In [26]:
snap_counts = nfl.load_snap_counts(seasons=SEASONS).to_pandas()
snap_counts.head()

,game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
0,2023_01_ARI_WAS,202309100was,2023,REG,1,Saahdiq Charles,CharSa00,G,WAS,ARI,71.0,1.0,0.0,0.0,4.0,0.14
1,2023_01_ARI_WAS,202309100was,2023,REG,1,Andrew Wylie,WyliAn00,T,WAS,ARI,71.0,1.0,0.0,0.0,4.0,0.14
2,2023_01_ARI_WAS,202309100was,2023,REG,1,Charles Leno Jr.,LenoCh00,T,WAS,ARI,71.0,1.0,0.0,0.0,4.0,0.14
3,2023_01_ARI_WAS,202309100was,2023,REG,1,Sam Howell,HoweSa00,QB,WAS,ARI,71.0,1.0,0.0,0.0,0.0,0.00
4,2023_01_ARI_WAS,202309100was,2023,REG,1,Nick Gates,GateNi00,C,WAS,ARI,71.0,1.0,0.0,0.0,0.0,0.00


In [27]:
print(f"snap_counts shape: {snap_counts.shape}")
snap_counts_summary = column_summary(snap_counts)
snap_counts_summary

snap_counts shape: (79767, 16)


,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,game_type,object,79767,79767,0,0.0,5,0.01,0.01,False,False,"[REG, WC, DIV, CON, SB]",categorical
1,season,int32,79767,79767,0,0.0,3,0.00,0.00,False,False,"[2023, 2024, 2025]",low_cardinality_numeric
2,week,int32,79767,79767,0,0.0,22,0.03,0.03,False,False,>10 values,numerical
3,st_snaps,float64,79767,79767,0,0.0,34,0.04,0.04,False,False,>10 values,numerical
4,defense_snaps,float64,79767,79767,0,0.0,94,0.12,0.12,False,False,>10 values,numerical
5,offense_snaps,float64,79767,79767,0,0.0,95,0.12,0.12,False,False,>10 values,numerical
6,st_pct,float64,79767,79767,0,0.0,98,0.12,0.12,False,False,>10 values,numerical
7,defense_pct,float64,79767,79767,0,0.0,101,0.13,0.13,False,False,>10 values,numerical
8,offense_pct,float64,79767,79767,0,0.0,101,0.13,0.13,False,False,>10 values,numerical
9,position,object,79767,79767,0,0.0,23,0.03,0.03,False,False,>10 values,text_or_high_cardinality_categorical


In [28]:
# check Candidate grain: PFR player ID, game, and team
snap_key = ["pfr_player_id", "game_id", "team"]
snap_duplicate_rows = snap_counts.duplicated(snap_key).sum()
snap_null_pfr_ids = snap_counts["pfr_player_id"].isna().sum()

display(
    pd.DataFrame(
        {
            "check": [
                "rows",
                "null pfr_player_id",
                "duplicate candidate grain",
                "minimum offense_snaps",
                "maximum offense_snaps",
            ],
            "value": [
                len(snap_counts),
                snap_null_pfr_ids,
                snap_duplicate_rows,
                snap_counts["offense_snaps"].min(),
                snap_counts["offense_snaps"].max(),
            ],
        }
    )
)

,check,value
0,rows,79767.0
1,null pfr_player_id,0.0
2,duplicate candidate grain,0.0
3,minimum offense_snaps,0.0
4,maximum offense_snaps,96.0


In [29]:
display(
    snap_counts.groupby("season")
    .agg(
        rows=("season", "size"),
        players=("pfr_player_id", "nunique"),
        teams=("team", "nunique"),
        min_week=("week", "min"),
        max_week=("week", "max"),
    )
    .reset_index()
)

,season,rows,players,teams,min_week,max_week
0,2023,26540,2145,32,1,22
1,2024,26615,2192,32,1,22
2,2025,26612,2189,32,1,22


In [30]:
# snap counts by position
display(snap_counts["position"].value_counts(dropna=False).head(20))

position
LB    11674
WR     8862
CB     8378
TE     5323
RB     5149
DE     5086
T      4977
DT     4529
G      4341
FS     2446
C      2376
SS     2172
QB     2163
S      1998
P      1715
LS     1710
K      1702
DL     1491
OL     1457
NT      981
Name: count, dtype: int64

In [31]:
# Match rate from pfr_player_id to the player master pfr_id
snaps_with_id = snap_counts.loc[snap_counts["pfr_player_id"].notna()]
snap_player_match_rate = snaps_with_id["pfr_player_id"].isin(
    players["pfr_id"].dropna()
).mean()

print(f"Snap counts → players PFR match rate: {snap_player_match_rate:.2%}")

Snap counts → players PFR match rate: 99.86%


## 6. Play-by-Play

### What we are verifying

- The size and width of one recent season
- Candidate grain: game and play
- Regular-season and postseason coverage
- Null game and play identifiers
- Team and week coverage
- Availability of receiver, air-yard, scoring, and field-position columns
- Receiver-ID match rate to the player master
- Whether future ingestion should process seasons separately

Only 2025 is loaded in this first pass to keep the exploration bounded.

In [32]:
pbp = nfl.load_pbp(seasons=LARGE_SOURCE_SEASONS).to_pandas()
pbp.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,...,id,fantasy_player_name,fantasy_player_id,fantasy,fantasy_id,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,None,None,None,None,NaN,2025-09-07,900.0,1800.0,...,None,None,None,None,None,0.0,0.0,-0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,NO,35.0,2025-09-07,900.0,1800.0,...,None,None,None,None,None,0.0,0.0,-0.352700,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,63.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,ARI,78.0,2025-09-07,896.0,1796.0,...,00-0033553,J.Conner,00-0033553,J.Conner,00-0033553,0.0,0.0,-0.190052,NaN,NaN,NaN,NaN,NaN,0.511128,-51.112807
3,85.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,ARI,75.0,2025-09-07,858.0,1758.0,...,00-0035228,T.McBride,00-0037744,T.McBride,00-0037744,1.0,0.0,1.317340,0.939998,4.750889,3.0,0.666726,0.43911,0.668940,33.105969
4,115.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,ARI,64.0,2025-09-07,820.0,1720.0,...,00-0035228,None,None,None,None,0.0,0.0,-1.694360,NaN,NaN,NaN,NaN,NaN,0.492038,50.796208


In [33]:
print(f"pbp shape: {pbp.shape}")
print(f"pbp pandas memory: {pbp.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")
pbp_summary = column_summary(pbp)
pbp_summary

pbp shape: (48771, 372)
pbp pandas memory: 370.14 MB


,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,end_yard_line,object,48771,0,48771,100.00,0,0.00,0.00,False,False,[],binary_categorical
1,lateral_kickoff_returner_player_id,object,48771,0,48771,100.00,0,0.00,0.00,False,False,[],binary_categorical
2,lateral_kickoff_returner_player_name,object,48771,0,48771,100.00,0,0.00,0.00,False,False,[],binary_categorical
3,lateral_punt_returner_player_id,object,48771,0,48771,100.00,0,0.00,0.00,False,False,[],binary_categorical
4,lateral_punt_returner_player_name,object,48771,0,48771,100.00,0,0.00,0.00,False,False,[],binary_categorical
...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,yrdln,object,48771,48417,354,0.73,1569,3.24,3.22,False,False,>10 values,text_or_high_cardinality_categorical
368,drive_real_start_time,object,48771,48199,572,1.17,6034,12.52,12.37,False,False,>10 values,text_or_high_cardinality_categorical
369,end_clock_time,object,48771,40315,8456,17.34,40313,100.00,82.66,False,False,>10 values,text_or_high_cardinality_categorical
370,desc,object,48771,48771,0,0.00,45768,93.84,93.84,False,False,>10 values,text_or_high_cardinality_categorical


In [34]:
# check nulls and candidate grain: game and play
pbp_key = ["game_id", "play_id"]
pbp_duplicate_rows = pbp.duplicated(pbp_key).sum()

display(
    pd.DataFrame(
        {
            "check": [
                "rows",
                "null game_id",
                "null play_id",
                "duplicate candidate grain",
            ],
            "value": [
                len(pbp),
                pbp["game_id"].isna().sum(),
                pbp["play_id"].isna().sum(),
                pbp_duplicate_rows,
            ],
        }
    )
)

,check,value
0,rows,48771
1,null game_id,0
2,null play_id,0
3,duplicate candidate grain,0


In [35]:
display(
    pbp.groupby(["season", "season_type"], dropna=False)
    .agg(
        rows=("season", "size"),
        games=("game_id", "nunique"),
        posteams=("posteam", "nunique"),
        min_week=("week", "min"),
        max_week=("week", "max"),
    )
    .reset_index()
)

,season,season_type,rows,games,posteams,min_week,max_week
0,2025,POST,2319,13,14,19,22
1,2025,REG,46452,272,32,1,18


In [36]:
# check availability of receiver, air-yard, scoring, and field-position columns
pbp_wr_columns = [
    "season",
    "week",
    "season_type",
    "game_id",
    "play_id",
    "posteam",
    "pass_attempt",
    "complete_pass",
    "receiver_player_id",
    "receiver_player_name",
    "air_yards",
    "receiving_yards",
    "yards_gained",
    "yardline_100",
    "goal_to_go",
    "touchdown",
    "pass_touchdown",
]

missing_pbp_wr_columns = [
    column for column in pbp_wr_columns if column not in pbp.columns
]
print("Missing candidate PBP WR columns:", missing_pbp_wr_columns)

Missing candidate PBP WR columns: []


In [37]:
# sample of data with selected columns 
display(
    pbp.loc[
        pbp["receiver_player_id"].notna(),
        [column for column in pbp_wr_columns if column in pbp.columns],
    ].head(15)
)

,season,week,season_type,game_id,play_id,posteam,pass_attempt,complete_pass,receiver_player_id,receiver_player_name,air_yards,receiving_yards,yards_gained,yardline_100,goal_to_go,touchdown,pass_touchdown
3,2025,1,REG,2025_01_ARI_NO,85.0,ARI,1.0,1.0,00-0037744,T.McBride,3.0,11.0,11.0,75.0,0.0,0.0,0.0
9,2025,1,REG,2025_01_ARI_NO,243.0,NO,1.0,0.0,00-0037545,R.Shaheed,2.0,NaN,0.0,74.0,0.0,0.0,0.0
12,2025,1,REG,2025_01_ARI_NO,327.0,ARI,1.0,1.0,00-0037744,T.McBride,-3.0,5.0,5.0,66.0,0.0,0.0,0.0
20,2025,1,REG,2025_01_ARI_NO,528.0,ARI,1.0,1.0,00-0038559,Mi.Wilson,3.0,5.0,5.0,27.0,0.0,0.0,0.0
22,2025,1,REG,2025_01_ARI_NO,575.0,ARI,1.0,0.0,00-0039041,E.Higgins,4.0,NaN,0.0,19.0,0.0,0.0,0.0
24,2025,1,REG,2025_01_ARI_NO,622.0,ARI,1.0,0.0,00-0038559,Mi.Wilson,22.0,NaN,0.0,24.0,0.0,0.0,0.0
27,2025,1,REG,2025_01_ARI_NO,695.0,NO,1.0,1.0,00-0036040,J.Johnson,5.0,11.0,11.0,75.0,0.0,0.0,0.0
29,2025,1,REG,2025_01_ARI_NO,742.0,NO,1.0,1.0,00-0037239,C.Olave,13.0,13.0,13.0,62.0,0.0,0.0,0.0
38,2025,1,REG,2025_01_ARI_NO,1017.0,NO,1.0,1.0,00-0031236,B.Cooks,3.0,12.0,12.0,37.0,0.0,0.0,0.0
40,2025,1,REG,2025_01_ARI_NO,1062.0,NO,1.0,1.0,00-0033906,A.Kamara,-5.0,7.0,7.0,25.0,0.0,0.0,0.0


In [38]:
# Receiver-ID match rate to the player master
pbp_receivers = pbp.loc[pbp["receiver_player_id"].notna()]
pbp_receiver_match_rate = pbp_receivers["receiver_player_id"].isin(
    players["gsis_id"]
).mean()

print(f"PBP receivers → players GSIS match rate: {pbp_receiver_match_rate:.2%}")

PBP receivers → players GSIS match rate: 100.00%


## 7. Participation

### What we are verifying

- Whether current participation data is available
- Observed game/play grain
- Representation of players within each row
- Available offense and defense position context
- Compatibility of participation game/play identifiers with play-by-play
- Whether this source is necessary for the first WR slice

Only 2025 is loaded in this first pass.

In [39]:
participation = nfl.load_participation(
    seasons=LARGE_SOURCE_SEASONS
).to_pandas()

participation.head()

,nflverse_game_id,old_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,players_on_play,offense_players,defense_players,n_offense,n_defense,ngs_air_yards,time_to_throw,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,2025_01_DAL_PHI,2025090400,40.0,DAL,None,"2 CB, 1 FB, 1 FS, 1 ILB, 2 OLB, 1 SS, 1 TE, 2 WR",0.0,"2 CB, 1 FB, 2 ILB, 1 K, 2 OLB, 1 RB, 1 SS, 1 TE",0.0,00-0039140;00-0039841;00-0031357;00-0040251;00-0038489;00-0038738;00-0037578;00-0039826;00-00400...,00-0031357;00-0040251;00-0038738;00-0037578;00-0036870;00-0037281;00-0039352;00-0037561;00-00385...,00-0039140;00-0039841;00-0038489;00-0039826;00-0040013;00-0033787;00-0036159;00-0036922;00-00397...,11,11,NaN,NaN,False,,,None,C.J. Goodwin;Trikweze Bridges;Hunter Luepke;Juanyeh Thomas;Buddy Johnson;Damone Clark;Marist Liu...,Kelee Ringo;Cooper DeJean;Ben VanSumeren;Jeremiah Trotter Jr.;Smael Mondon Jr.;Jake Elliott;Josh...,CB;CB;FB;FS;ILB;OLB;OLB;SS;TE;WR;WR,CB;CB;FB;ILB;ILB;K;OLB;OLB;RB;SS;TE,29;25;40;2;32;18;35;14;86;9;19,7;33;43;54;42;4;0;48;28;21;83
1,2025_01_DAL_PHI,2025090400,71.0,DAL,UNDER CENTER,"1 C, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR",6.0,"3 CB, 2 DT, 1 FS, 2 ILB, 2 OLB, 1 SS",0.0,00-0039348;00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0037243;00-00400...,00-0039348;00-0037243;00-0040007;00-0033077;00-0036997;00-0036036;00-0039341;00-0038041;00-00363...,00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,11,11,NaN,NaN,False,,,None,Cooper Beebe;Tyler Smith;Tyler Booker;Dak Prescott;Javonte Williams;Terence Steele;Tyler Guyton;...,Adoree' Jackson;Quinyon Mitchell;Cooper DeJean;Jordan Davis;Moro Ojomo;Reed Blankenship;Zack Bau...,C;G;G;QB;RB;T;T;TE;WR;WR;WR,CB;CB;CB;DT;DT;FS;ILB;ILB;OLB;OLB;SS,56;73;52;4;33;78;60;87;88;3;1,8;27;33;90;97;32;53;30;3;58;24
2,2025_01_DAL_PHI,2025090400,112.0,DAL,SHOTGUN,"1 C, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR",6.0,"3 CB, 2 DT, 1 FS, 2 ILB, 2 OLB, 1 SS",0.0,00-0039348;00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0037243;00-00400...,00-0039348;00-0037243;00-0040007;00-0033077;00-0036997;00-0036036;00-0039341;00-0038041;00-00363...,00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,11,11,NaN,NaN,False,,,None,Cooper Beebe;Tyler Smith;Tyler Booker;Dak Prescott;Javonte Williams;Terence Steele;Tyler Guyton;...,Adoree' Jackson;Quinyon Mitchell;Cooper DeJean;Jordan Davis;Moro Ojomo;Reed Blankenship;Zack Bau...,C;G;G;QB;RB;T;T;TE;WR;WR;WR,CB;CB;CB;DT;DT;FS;ILB;ILB;OLB;OLB;SS,56;73;52;4;33;78;60;87;88;9;3,8;27;33;90;97;32;53;30;3;58;24
3,2025_01_DAL_PHI,2025090400,141.0,DAL,SHOTGUN,"1 C, 1 FB, 2 G, 1 QB, 2 T, 1 TE, 3 WR",7.0,"2 CB, 3 DT, 1 FS, 2 ILB, 2 OLB, 1 SS",4.0,00-0039348;00-0039888;00-0039841;00-0037073;00-0038978;00-0038412;00-0038738;00-0037130;00-00372...,00-0039348;00-0038738;00-0037243;00-0040007;00-0033077;00-0036036;00-0039341;00-0038547;00-00363...,00-0039888;00-0039841;00-0037073;00-0038978;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,11,11,NaN,1.6,False,HITCH/CURL,ZONE_COVERAGE,COVER_2,Cooper Beebe;Hunter Luepke;Tyler Smith;Tyler Booker;Dak Prescott;Terence Steele;Tyler Guyton;Luk...,Quinyon Mitchell;Cooper DeJean;Jordan Davis;Byron Young;Moro Ojomo;Reed Blankenship;Zack Baun;Ji...,C;FB;G;G;QB;T;T;TE;WR;WR;WR,CB;CB;DT;DT;DT;FS;ILB;ILB;OLB;OLB;SS,56;40;73;52;4;78;60;86;88;9;3,27;33;90;94;97;32;53;30;3;58;21
4,2025_01_DAL_PHI,2025090400,166.0,DAL,UNDER CENTER,"1 C, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR",8.0,"3 CB, 2 DT, 1 FS, 2 ILB, 2 OLB, 1 SS",0.0,00-0039348;00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0037243;00-00400...,00-0039348;00-0037243;00-0040007;00-0033077;00-0036997;00-0036036;00-0039341;00-0039530;00-00363...,00-0033878;00-0039888;00-0039841;00-0037073;00-0038412

In [40]:
participation_summary = column_summary(participation)
participation_summary

,column,dtype,n_rows,n_non_null,n_missing,pct_missing,n_unique,pct_unique_non_null,pct_unique_total,is_constant,maybe_id,sample_unique_values,suggested_type
0,was_pressure,object,45184,45175,9,0.02,2,0.00,0.00,False,False,"[False, True]",binary_categorical
1,ngs_air_yards,float64,45184,0,45184,100.00,0,0.00,0.00,False,False,[],binary_numeric
2,defense_man_zone_type,object,45184,45175,9,0.02,3,0.01,0.01,False,False,"[, ZONE_COVERAGE, MAN_COVERAGE]",categorical
3,offense_formation,object,45184,36076,9108,20.16,3,0.01,0.01,False,False,"[UNDER CENTER, SHOTGUN, PISTOL]",categorical
4,defense_coverage_type,object,45184,22055,23129,51.19,10,0.05,0.02,False,False,"[COVER_2, COVER_1, COVER_0, COVER_3, COVER_9, 2_MAN, COVER_6, COVER_4, COMBO, BLOWN]",categorical
5,route,object,45184,45175,9,0.02,14,0.03,0.03,False,False,>10 values,categorical
6,n_offense,int32,45184,45184,0,0.00,5,0.01,0.01,False,False,"[11, 12, 10, 14, 16]",low_cardinality_numeric
7,n_defense,int32,45184,45184,0,0.00,6,0.01,0.01,False,False,"[11, 12, 10, 13, 14, 9]",low_cardinality_numeric
8,number_of_pass_rushers,float64,45184,45175,9,0.02,10,0.02,0.02,False,False,"[0.0, 4.0, 5.0, 3.0, 6.0, 7.0, 8.0, 2.0, 1.0, 9.0]",low_cardinality_numeric
9,defenders_in_box,float64,45184,45175,9,0.02,12,0.03,0.03,False,False,>10 values,low_cardinality_numeric


In [41]:
# check nulls and observed game/play grain
participation_key = ["nflverse_game_id", "play_id"]
participation_duplicate_rows = participation.duplicated(participation_key).sum()

display(
    pd.DataFrame(
        {
            "check": [
                "rows",
                "null nflverse_game_id",
                "null play_id",
                "duplicate candidate grain",
            ],
            "value": [
                len(participation),
                participation["nflverse_game_id"].isna().sum(),
                participation["play_id"].isna().sum(),
                participation_duplicate_rows,
            ],
        }
    )
)

,check,value
0,rows,45184
1,null nflverse_game_id,0
2,null play_id,0
3,duplicate candidate grain,0


In [42]:
# sample with selected columns
participation_columns = [
    "nflverse_game_id",
    "old_game_id",
    "play_id",
    "possession_team",
    "players_on_play",
    "offense_players",
    "defense_players",
    "offense_positions",
    "defense_positions",
]

display(
    participation[
        [column for column in participation_columns if column in participation.columns]
    ].head()
)

,nflverse_game_id,old_game_id,play_id,possession_team,players_on_play,offense_players,defense_players,offense_positions,defense_positions
0,2025_01_DAL_PHI,2025090400,40.0,DAL,00-0039140;00-0039841;00-0031357;00-0040251;00-0038489;00-0038738;00-0037578;00-0039826;00-00400...,00-0031357;00-0040251;00-0038738;00-0037578;00-0036870;00-0037281;00-0039352;00-0037561;00-00385...,00-0039140;00-0039841;00-0038489;00-0039826;00-0040013;00-0033787;00-0036159;00-0036922;00-00397...,CB;CB;FB;FS;ILB;OLB;OLB;SS;TE;WR;WR,CB;CB;FB;ILB;ILB;K;OLB;OLB;RB;SS;TE
1,2025_01_DAL_PHI,2025090400,71.0,DAL,00-0039348;00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0037243;00-00400...,00-0039348;00-0037243;00-0040007;00-0033077;00-0036997;00-0036036;00-0039341;00-0038041;00-00363...,00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,C;G;G;QB;RB;T;T;TE;WR;WR;WR,CB;CB;CB;DT;DT;FS;ILB;ILB;OLB;OLB;SS
2,2025_01_DAL_PHI,2025090400,112.0,DAL,00-0039348;00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0037243;00-00400...,00-0039348;00-0037243;00-0040007;00-0033077;00-0036997;00-0036036;00-0039341;00-0038041;00-00363...,00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,C;G;G;QB;RB;T;T;TE;WR;WR;WR,CB;CB;CB;DT;DT;FS;ILB;ILB;OLB;OLB;SS
3,2025_01_DAL_PHI,2025090400,141.0,DAL,00-0039348;00-0039888;00-0039841;00-0037073;00-0038978;00-0038412;00-0038738;00-0037130;00-00372...,00-0039348;00-0038738;00-0037243;00-0040007;00-0033077;00-0036036;00-0039341;00-0038547;00-00363...,00-0039888;00-0039841;00-0037073;00-0038978;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,C;FB;G;G;QB;T;T;TE;WR;WR;WR,CB;CB;DT;DT;DT;FS;ILB;ILB;OLB;OLB;SS
4,2025_01_DAL_PHI,2025090400,166.0,DAL,00-0039348;00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0037243;00-00400...,00-0039348;00-0037243;00-0040007;00-0033077;00-0036997;00-0036036;00-0039341;00-0039530;00-00363...,00-0033878;00-0039888;00-0039841;00-0037073;00-0038412;00-0037130;00-0036418;00-0040708;00-00390...,C;G;G;QB;RB;T;T;TE;WR;WR;WR,CB;CB;CB;DT;DT;FS;ILB;ILB;OLB;OLB;SS


In [43]:
participation_game_match_rate = participation["nflverse_game_id"].isin(
    pbp["game_id"]
).mean()

participation_play_keys = participation[
    ["nflverse_game_id", "play_id"]
].rename(columns={"nflverse_game_id": "game_id"})

pbp_play_keys = pbp[["game_id", "play_id"]].drop_duplicates()

participation_play_match_rate = participation_play_keys.merge(
    pbp_play_keys,
    on=["game_id", "play_id"],
    how="left",
    indicator=True,
)["_merge"].eq("both").mean()

print(f"Participation games → PBP game match rate: {participation_game_match_rate:.2%}")
print(f"Participation game/play → PBP match rate: {participation_play_match_rate:.2%}")

Participation games → PBP game match rate: 100.00%
Participation game/play → PBP match rate: 100.00%


## Results

## 8. Cross-source checks

These checks summarize the joins and grains that matter most for the first WR slice.

In [44]:
cross_source_summary = pd.DataFrame(
    [
        {
            "check": "player_stats player_id → players gsis_id",
            "result": player_stats_player_match_rate,
            "format": "match rate",
        },
        {
            "check": "rosters_weekly gsis_id → players gsis_id",
            "result": roster_player_match_rate,
            "format": "match rate",
        },
        {
            "check": "snap_counts pfr_player_id → players pfr_id",
            "result": snap_player_match_rate,
            "format": "match rate",
        },
        {
            "check": "pbp receiver_player_id → players gsis_id",
            "result": pbp_receiver_match_rate,
            "format": "match rate",
        },
        {
            "check": "participation game/play → pbp game/play",
            "result": participation_play_match_rate,
            "format": "match rate",
        },
    ]
)

cross_source_summary["percent"] = cross_source_summary["result"].map(
    lambda value: f"{value:.2%}"
)
display(cross_source_summary[["check", "percent"]])

,check,percent
0,player_stats player_id → players gsis_id,100.00%
1,rosters_weekly gsis_id → players gsis_id,99.59%
2,snap_counts pfr_player_id → players pfr_id,99.86%
3,pbp receiver_player_id → players gsis_id,100.00%
4,participation game/play → pbp game/play,100.00%


In [45]:
grain_summary = pd.DataFrame(
    [
        {
            "dataset": "players",
            "candidate_grain": "gsis_id",
            "duplicate_rows": players.loc[players["gsis_id"].notna(), "gsis_id"].duplicated().sum(),
            "null_key_rows": players["gsis_id"].isna().sum(),
        },
        {
            "dataset": "schedules",
            "candidate_grain": "game_id",
            "duplicate_rows": schedules["game_id"].duplicated().sum(),
            "null_key_rows": schedules["game_id"].isna().sum(),
        },
        {
            "dataset": "player_stats",
            "candidate_grain": "player_id + season + week + season_type",
            "duplicate_rows": player_stats_duplicate_rows,
            "null_key_rows": player_stats[player_stats_key].isna().any(axis=1).sum(),
        },
        {
            "dataset": "rosters_weekly",
            "candidate_grain": "gsis_id + season + week + team",
            "duplicate_rows": roster_duplicate_rows,
            "null_key_rows": rosters_weekly[roster_key].isna().any(axis=1).sum(),
        },
        {
            "dataset": "snap_counts",
            "candidate_grain": "pfr_player_id + game_id + team",
            "duplicate_rows": snap_duplicate_rows,
            "null_key_rows": snap_counts[snap_key].isna().any(axis=1).sum(),
        },
        {
            "dataset": "pbp",
            "candidate_grain": "game_id + play_id",
            "duplicate_rows": pbp_duplicate_rows,
            "null_key_rows": pbp[pbp_key].isna().any(axis=1).sum(),
        },
        {
            "dataset": "participation",
            "candidate_grain": "nflverse_game_id + play_id",
            "duplicate_rows": participation_duplicate_rows,
            "null_key_rows": participation[participation_key].isna().any(axis=1).sum(),
        },
    ]
)

display(grain_summary)

,dataset,candidate_grain,duplicate_rows,null_key_rows
0,players,gsis_id,0,0
1,schedules,game_id,0,0
2,player_stats,player_id + season + week + season_type,0,66
3,rosters_weekly,gsis_id + season + week + team,0,30
4,snap_counts,pfr_player_id + game_id + team,0,0
5,pbp,game_id + play_id,0,0
6,participation,nflverse_game_id + play_id,0,0


## 9. WR source map

This is the initial source choice for each WR concept. It is intentionally manual and concise; exact definitions belong in Silver and Gold notebooks.

In [46]:
wr_source_map = pd.DataFrame(
    [
        ["Player identity", "players / weekly rosters", "gsis_id; reconcile weekly team and position"],
        ["Games and weeks", "schedules", "game_id, season, week, game_type"],
        ["Targets", "weekly player stats", "Validate against PBP before treating as authoritative"],
        ["Receptions", "weekly player stats", "Validate against PBP"],
        ["Receiving yards", "weekly player stats", "Validate against PBP"],
        ["Receiving touchdowns", "weekly player stats", "Validate against PBP"],
        ["Air yards", "weekly player stats", "Use PBP for play-level context"],
        ["Team targets", "play-by-play", "Requires a documented Silver pass/target classification"],
        ["Team pass attempts", "play-by-play", "Requires a documented Silver attempt classification"],
        ["Red-zone targets", "play-by-play", "Requires a yardline and target definition"],
        ["End-zone targets", "play-by-play", "Requires an end-zone target definition"],
        ["Offensive snaps", "snap counts", "Requires PFR-to-GSIS crosswalk validation"],
        ["Play participation", "participation", "Conditional; list-valued player fields"],
    ],
    columns=["WR concept", "candidate source", "remaining question"],
)

display(wr_source_map)

,WR concept,candidate source,remaining question
0,Player identity,players / weekly rosters,gsis_id; reconcile weekly team and position
1,Games and weeks,schedules,"game_id, season, week, game_type"
2,Targets,weekly player stats,Validate against PBP before treating as authoritative
3,Receptions,weekly player stats,Validate against PBP
4,Receiving yards,weekly player stats,Validate against PBP
5,Receiving touchdowns,weekly player stats,Validate against PBP
6,Air yards,weekly player stats,Use PBP for play-level context
7,Team targets,play-by-play,Requires a documented Silver pass/target classification
8,Team pass attempts,play-by-play,Requires a documented Silver attempt classification
9,Red-zone targets,play-by-play,Requires a yardline and target definition


## Takeaways

- Weekly player stats, players, schedules, weekly rosters, and play-by-play are the core Bronze sources for the first WR slice.
- Snap counts are useful if the PFR-to-GSIS crosswalk is sufficiently complete.
- Participation should remain optional until a specific WR metric requires its list-valued play-level data.
- Bronze should preserve null-ID source rows; Silver should decide how to classify or exclude them.
- Play-by-play should be ingested one season at a time because its pandas footprint is materially larger than the other sources.
- The next notebook should persist the confirmed source data as Parquet without introducing fantasy feature engineering.